In [31]:
import os, json, math, warnings, random
from typing import Dict, List, Tuple, Optional, Callable
from collections import defaultdict, Counter

import numpy as np
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as models

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
def presence_f1_from_logits(logits_p: torch.Tensor, gt_presence: torch.Tensor, thresh: float = 0.5, eps: float = 1e-6) -> float:
    """Macro F1 over part presence channels.
    logits_p: (B, P)
    gt_presence: (B, P) binary {0,1}
    """
    with torch.no_grad():
        prob = torch.sigmoid(logits_p)
        pred = (prob >= thresh).float()
        tp = (pred * gt_presence).sum(dim=0)
        fp = (pred * (1 - gt_presence)).sum(dim=0)
        fn = ((1 - pred) * gt_presence).sum(dim=0)
        precision = tp / (tp + fp + eps)
        recall    = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        # macro over parts (only parts that appear in dataset? we macro over all channels for stability)
        return float(f1.mean().item())

In [33]:
PART_VOCAB = [
    "head","body","foot","hand","tail","wing","fin",
    "engine","tire","seat","sail","side_mirror","mouth"
]
PART_TO_IDX = {p:i for i,p in enumerate(PART_VOCAB)}

# Regex-like matching (string contains) to map category names → canonical types
# tolerant to typos like "Tier" for "Tire".
CANON_RULES = [
    ("side mirror", "side_mirror"),
    ("tier", "tire"), ("tire", "tire"), ("tyre", "tire"),
    ("engine", "engine"),
    ("sail", "sail"),
    ("mouth", "mouth"),
    ("seat", "seat"),
    ("wing", "wing"),
    ("fin", "fin"),
    ("head", "head"),
    ("body", "body"),
    ("hand", "hand"),
    ("foot", "foot"),
    ("tail", "tail"),
]

def match_part_type(name: str) -> Optional[str]:
    n = name.lower()
    for token, canon in CANON_RULES:
        if token in n:
            return canon
    return None

# Weak prior for presence when JSON lacks pixel parts but has supercategory
SUPER_TO_PARTS = {
    # animals
    "Quadruped": {"head","body","foot","tail"},
    "Biped": {"head","body","hand","foot","tail"},  # tail appears in your list
    "Bird": {"head","body","wing","foot","tail"},
    "Fish": {"head","body","fin","tail","mouth"},
    "Snake": {"head","body","tail"},
    "Reptile": {"head","body","foot","tail"},
    # artifacts / vehicles
    "Car": {"body","tire","side_mirror"},
    "Bicycle": {"body","head","seat","tire"},
    "Boat": {"body","sail"},
    "Aeroplane": {"head","body","engine","wing","tail"},
    "Bottle": {"mouth","body"},
}

In [34]:
class PartImageNetCOCODataset(Dataset):
    """
    Expects COCO-style JSON where each annotation corresponds to a *part instance*.
    The specific part is given by `category_id`, and the categories array
    contains entries like {id, name: "Quadruped Head", supercategory: "Quadruped"}.

    We build a shared part space (channels) by mapping category *names* to
    canonical types (head/body/wing/...). If the JSON actually contains object
    categories (e.g., ImageNet synsets) and no part tokens, we fallback to
    weak presence labels derived from supercategory.
    """
    def __init__(self,
                 ann_path: str,
                 img_root: str,
                 img_size: Tuple[int,int]=(224,224),
                 augment: bool=True,
                 shared_by_type: bool=True):
        super().__init__()
        self.ann_path = ann_path
        self.img_root = img_root
        self.W, self.H = img_size[0], img_size[1]
        self.augment = augment
        self.shared_by_type = shared_by_type

        with open(ann_path, 'r') as f:
            data = json.load(f)

        # Index images, annotations, categories
        self.images: Dict[int,dict] = {im['id']: im for im in data['images']}
        self.img_ids: List[int] = list(self.images.keys())
        self.categories: Dict[int,dict] = {c['id']: c for c in data['categories']}

        self.anns_by_img: Dict[int,List[dict]] = defaultdict(list)
        for a in data['annotations']:
            self.anns_by_img[a['image_id']].append(a)

        # Build label space (supercategories)
        self.supercats = sorted({c.get('supercategory', 'unknown') for c in self.categories.values()})
        self.super_to_idx = {s:i for i,s in enumerate(self.supercats)}

        # Build shared part-type channels from category *names*
        part_types = []
        for c in self.categories.values():
            ptype = match_part_type(c['name'])
            if ptype and ptype not in part_types:
                part_types.append(ptype)
        part_types.sort()
        self.part_types = part_types  # actual seen types in this split
        self.num_part_channels = len(part_types) if shared_by_type else len(self.categories)

        # Map category_id → channel index
        self.channel_of_cat: Dict[int, int] = {}
        if shared_by_type:
            for cid, c in self.categories.items():
                ptype = match_part_type(c['name'])
                if ptype is not None and ptype in self.part_types:
                    self.channel_of_cat[cid] = self.part_types.index(ptype)
        else:
            for cid in self.categories:
                self.channel_of_cat[cid] = cid  # one channel per category

        # Heuristic: detect if *any* part polygons are likely present
        # If none of the category names matched part tokens, treat as object-only
        self.has_part_tokens = (len(self.part_types) > 0)

        print(f"[init] ann_path={ann_path}")
        print(f"[init] images={len(self.img_ids)} | categories={len(self.categories)}")
        print(f"[init] Recognized part types (shared): {self.part_types}")
        print(f"[init] Shared-by-type part channels: {self.num_part_channels}; Total input channels: {3 + self.num_part_channels}")

        # Transforms
        base = [T.Resize((self.H, self.W))]
        if augment:
            base += [T.RandomHorizontalFlip(p=0.5), T.RandomRotation(degrees=10)]
        base += [T.ToTensor()]
        self.img_tf = T.Compose(base)

    def __len__(self):
        return len(self.img_ids)

    def _image_path(self, info: dict) -> str:
        # JSON typically stores file_name relative to split root, e.g. "n014xxx/JPEG..."
        return os.path.join(self.img_root, info['file_name'])

    def _draw_polygon(self, draw: ImageDraw.ImageDraw, poly: list, info: dict):
        # `poly` is a flat list [x0,y0, x1,y1, ...] in original image coordinates
        xs, ys = poly[0::2], poly[1::2]
        ow, oh = info['width'], info['height']
        pts = []
        for x, y in zip(xs, ys):
            xi = int(round(x / ow * self.W))
            yi = int(round(y / oh * self.H))
            pts.append((xi, yi))
        if len(pts) >= 3:
            draw.polygon(pts, outline=1, fill=1)

    def _rasterize_masks(self, iid: int) -> torch.Tensor:
        """Return (C,H,W) mask tensor in {0,1}. If no recognized part tokens,
        returns zeros (we'll use weak presence by supercategory).
        """
        C = self.num_part_channels
        mask_imgs = [Image.new('L', (self.W, self.H), 0) for _ in range(C)]
        draws = [ImageDraw.Draw(mi) for mi in mask_imgs]

        anns = self.anns_by_img.get(iid, [])
        if len(anns) == 0 or C == 0:
            return torch.zeros((C, self.H, self.W), dtype=torch.float32)

        for ann in anns:
            cid = int(ann['category_id'])
            ch = self.channel_of_cat.get(cid)
            if ch is None:
                continue
            seg = ann.get('segmentation', None)
            if isinstance(seg, list) and len(seg) > 0:
                for poly in seg:
                    if isinstance(poly, list) and len(poly) >= 6:
                        self._draw_polygon(draws[ch], poly, self.images[iid])
            else:
                # bbox fallback if segmentation missing
                x, y, w, h = ann['bbox']
                info = self.images[iid]
                ow, oh = info['width'], info['height']
                xs = int(round(x / ow * self.W))
                ys = int(round(y / oh * self.H))
                xe = int(round((x + w) / ow * self.W))
                ye = int(round((y + h) / oh * self.H))
                draws[ch].rectangle([xs, ys, xe, ye], outline=1, fill=1)

        mask = torch.stack([TF.to_tensor(mi).squeeze(0) for mi in mask_imgs], dim=0).to(torch.float32)
        return mask

    def __getitem__(self, idx: int):
        iid = self.img_ids[idx]
        info = self.images[iid]
        img_path = self._image_path(info)
        if not os.path.exists(img_path):
            # Safe fail: return None, collate will drop
            return None

        img = Image.open(img_path).convert('RGB')
        img = self.img_tf(img)  # (3,H,W)

        # Label (supercategory of *first* annotation)
        anns = self.anns_by_img.get(iid, [])
        if len(anns) == 0:
            label_idx = 0
            super_name = self.supercats[0]
        else:
            first_cid = int(anns[0]['category_id'])
            super_name = self.categories[first_cid].get('supercategory', 'unknown')
            label_idx = self.super_to_idx.get(super_name, 0)

        # Pixel masks (shared)
        mask = self._rasterize_masks(iid) if self.has_part_tokens else torch.zeros((self.num_part_channels, self.H, self.W))

        # Part presence target
        if self.has_part_tokens:
            presence = (mask.sum(dim=(1,2)) > 0).float()
            has_pix_flag = float(presence.sum().item() > 0)
        else:
            presence = torch.zeros(len(PART_VOCAB), dtype=torch.float32)
            for p in SUPER_TO_PARTS.get(super_name, set()):
                presence[PART_TO_IDX[p]] = 1.0
            # If dataset shares only the recognized part_types subset, crop presence
            if self.num_part_channels != len(PART_VOCAB):
                # Build mapping vector from PART_VOCAB → part_types subset
                pres_sub = torch.zeros(self.num_part_channels, dtype=torch.float32)
                for i, p in enumerate(self.part_types):
                    pres_sub[i] = presence[PART_TO_IDX[p]]
                presence = pres_sub
            has_pix_flag = 0.0

        return img, torch.tensor(label_idx, dtype=torch.long), mask, presence, torch.tensor(has_pix_flag, dtype=torch.float32)


In [35]:
def collate_drop_none(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    imgs, labels, masks, presence, flags = zip(*batch)
    return (
        torch.stack(imgs, dim=0),
        torch.stack(labels, dim=0),
        torch.stack(masks, dim=0),
        torch.stack(presence, dim=0),
        torch.stack(flags, dim=0),
    )



In [36]:

class SharedMTLResNet(nn.Module):
    def __init__(self, num_labels: int, num_parts: int,
                 use_sam: bool=False, sam_predictor: Optional[Callable]=None):
        super().__init__()
        self.use_sam = use_sam
        self.sam_predictor = sam_predictor

        base = models.resnet18(weights=None)
        self.backbone = base

        # Replace first conv to accept 3 + num_parts channels
        in_ch = 3 + num_parts
        self.backbone.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)

        feat_dim = base.fc.in_features
        self.backbone.fc = nn.Identity()

        self.fc_label = nn.Linear(feat_dim, num_labels)
        self.fc_parts = nn.Linear(feat_dim, num_parts)   # part PRESENCE head (multi-label)

    def forward(self, x_rgb: torch.Tensor, x_masks: Optional[torch.Tensor]=None):
        if x_masks is None or x_masks.size(1) == 0:
            # Optional: call SAM to get pseudo part masks
            if self.use_sam and self.sam_predictor is not None:
                with torch.no_grad():
                    x_masks = self.sam_predictor(x_rgb)
            else:
                x_masks = torch.zeros(x_rgb.size(0), 0, x_rgb.size(2), x_rgb.size(3), device=x_rgb.device)
        x = torch.cat([x_rgb, x_masks], dim=1)
        feats = self.backbone(x)
        logits_y = self.fc_label(feats)
        logits_p = self.fc_parts(feats)
        return logits_y, logits_p

In [37]:
def part_presence_f1(prob: torch.Tensor, target: torch.Tensor, thresh: float=0.5) -> float:
    pred = (prob > thresh).int()
    tgt = target.int()
    tp = (pred & tgt).sum().item()
    fp = (pred & (1 - tgt)).sum().item()
    fn = ((1 - pred) & tgt).sum().item()
    if tp + fp + fn == 0:
        return 0.0
    return 2 * tp / (2 * tp + fp + fn)

class EarlyStopper:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -1e9
        self.count = 0
        self.stop = False
    def step(self, metric: float):
        if metric > self.best + self.min_delta:
            self.best = metric
            self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience:
                self.stop = True

In [38]:
def train_one_epoch(model, loader, opt):
    model.train()
    tot, corr = 0, 0
    f1s = []
    for step, batch in enumerate(loader):
        if batch is None:
            continue
        x, y, m, pres, has_pix = batch
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)

        logits_y, logits_p = model(x, m)
        loss_y = F.cross_entropy(logits_y, y)
        loss_p = F.binary_cross_entropy_with_logits(logits_p, pres)
        loss = loss_y + 0.1 * loss_p

        opt.zero_grad(); loss.backward(); opt.step()

        corr += (logits_y.argmax(1) == y).sum().item()
        tot += y.size(0)
        f1s.append(part_presence_f1(torch.sigmoid(logits_p).detach(), pres))

        if step % 100 == 0:
            # Diagnostics: how many channels are entirely zero in this batch's GT masks
            zero_ch = int((m.sum(dim=(0,2,3)) == 0).sum().item()) if m.numel() > 0 else 0
            print(f"[dbg] step {step} | gt zero-channels (count over batch*channels): {zero_ch}")

    acc = corr / max(1, tot)
    f1 = float(np.mean(f1s)) if len(f1s) else 0.0
    return acc, f1

def evaluate(model, loader):
    model.eval()
    tot, corr = 0, 0
    f1s = []
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            x, y, m, pres, has_pix = batch
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            logits_y, logits_p = model(x, m)
            corr += (logits_y.argmax(1) == y).sum().item()
            tot += y.size(0)
            f1s.append(part_presence_f1(torch.sigmoid(logits_p), pres))
    acc = corr / max(1, tot)
    f1 = float(np.mean(f1s)) if len(f1s) else 0.0
    return acc, f1

In [39]:
def audit_masks(ds: PartImageNetCOCODataset, max_images=200):
    n = min(max_images, len(ds))
    nz_images = 0
    nz_channels = set()
    for i in range(n):
        sample = ds[i]
        if sample is None:
            continue
        _, _, masks, presence, has_pix = sample
        if (masks.sum() > 0).item():
            nz_images += 1
        for ch in range(masks.size(0)):
            if masks[ch].sum().item() > 0:
                nz_channels.add(ch)
    print(f"[audit] images scanned: {n}")
    print(f"[audit] images with any foreground mask: {nz_images}")
    print(f"[audit] approx. part-channels with any pixels across scanned images: {len(nz_channels)}")


In [40]:
ROOT = "PartImageNet_Seg/PartImageNet"

TRAIN_JSON     = os.path.join(ROOT, "annotations/train/train.json")
VAL_JSON       = os.path.join(ROOT, "annotations/val/val.json")
TRAIN_IMG_ROOT = os.path.join(ROOT, "images/train")
VAL_IMG_ROOT   = os.path.join(ROOT, "images/val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = PartImageNetCOCODataset(
    ann_path=TRAIN_JSON,
    img_root=TRAIN_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=True,
    shared_by_type=True,
)
val_ds = PartImageNetCOCODataset(
    ann_path=VAL_JSON,
    img_root=VAL_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=False,
    shared_by_type=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          collate_fn=collate_drop_none, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          collate_fn=collate_drop_none, drop_last=False)

print("\nRunning audit on training data (first 200 images)…")
audit_masks(train_ds, max_images=200)

[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/train/train.json
[init] images=20481 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16
[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/val/val.json
[init] images=1206 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16

Running audit on training data (first 200 images)…
[audit] images scanned: 200
[audit] images with any foreground mask: 200
[audit] approx. part-channels with any pixels across scanned images: 3


## Independent Channel

In [41]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class IndepSpec:
    supers: List[str]
    parts: List[str]
    tuples: List[Tuple[str,str]]               # [(super, part), ...]
    tuple_to_idx: Dict[Tuple[str,str], int]
    part_to_shared: Dict[str, int]             # shared PART_VOCAB -> idx


def build_indep_spec(supercats: List[str]) -> IndepSpec:
    tuples: List[Tuple[str,str]] = []
    for s in supercats:
        for p in sorted(SUPER_TO_PARTS.get(s, set())):
            tuples.append((s, p))
    tuple_to_idx = {t:i for i,t in enumerate(tuples)}
    part_to_shared = {p: PART_TO_IDX[p] for p in PART_VOCAB}
    return IndepSpec(
        supers=list(supercats),
        parts=list(PART_VOCAB),
        tuples=tuples,
        tuple_to_idx=tuple_to_idx,
        part_to_shared=part_to_shared,
    )


def pack_shared_to_indep_batch(m_shared: torch.Tensor, pres_shared: torch.Tensor,
                               super_idx: torch.Tensor, spec: IndepSpec,
                               super_list: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Map shared (B,P,H,W) & (B,P) → independent (B,K,H,W) & (B,K).
    Only the channels corresponding to the sample’s supercategory are populated.
    Everything else remains zeros.
    """
    B, P, H, W = m_shared.shape
    K = len(spec.tuples)
    m_ind = torch.zeros(B, K, H, W, device=m_shared.device, dtype=m_shared.dtype)
    y_ind = torch.zeros(B, K, device=pres_shared.device, dtype=pres_shared.dtype)

    for b in range(B):
        sname = super_list[int(super_idx[b].item())]
        # for each part allowed under this super, copy over the shared channel
        for p in SUPER_TO_PARTS.get(sname, set()):
            k = spec.tuple_to_idx[(sname, p)]
            ps = spec.part_to_shared[p]
            m_ind[b, k] = m_shared[b, ps]
            y_ind[b, k] = pres_shared[b, ps]
    return m_ind, y_ind


class IndependentMTLResNet(nn.Module):
    """Multitask ResNet that consumes INDEPENDENT channels and predicts:
    - label logits over supercategories
    - part-presence logits over independent (super,part) tuples
    """
    def __init__(self, num_labels: int, num_indep_parts: int):
        super().__init__()
        base = models.resnet18(weights=None)
        in_ch = 3 + num_indep_parts
        base.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
        feat_dim = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.fc_label = nn.Linear(feat_dim, num_labels)
        self.fc_ind   = nn.Linear(feat_dim, num_indep_parts)

    def forward(self, x_rgb: torch.Tensor, x_mind: torch.Tensor):
        x = torch.cat([x_rgb, x_mind], dim=1)
        feats = self.backbone(x)
        return self.fc_label(feats), self.fc_ind(feats)

In [42]:
def train_one_epoch_indep(model: nn.Module, loader, spec: IndepSpec, super_list: List[str], opt) -> Tuple[float,float]:
    model.train(); tot, corr = 0, 0; f1s = []
    for step, (x, y, m_shared, pres_shared, haspix) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        m_shared, pres_shared = m_shared.to(device), pres_shared.to(device)
        m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, super_list)
        logits_y, logits_i = model(x, m_ind)
        loss_y = F.cross_entropy(logits_y, y)
        loss_i = F.binary_cross_entropy_with_logits(logits_i, y_ind)
        loss = loss_y + 0.1 * loss_i
        opt.zero_grad(); loss.backward(); opt.step()
        corr += (logits_y.argmax(1) == y).sum().item(); tot += y.size(0)
        f1s.append(presence_f1_from_logits(logits_i, y_ind))
        if step % 100 == 0:
            zeros = int((m_ind.sum(dim=(0,2,3)) == 0).sum().item())
            print(f"[indep] step {step} | zero indep-channels across batch: {zeros}/{m_ind.size(1)}")
    return corr/max(1,tot), float(np.mean(f1s))


def evaluate_indep(model: nn.Module, loader, spec: IndepSpec, super_list: List[str]) -> Tuple[float,float]:
    model.eval(); tot, corr = 0, 0; f1s = []
    with torch.no_grad():
        for (x, y, m_shared, pres_shared, haspix) in loader:
            x, y = x.to(device), y.to(device)
            m_shared, pres_shared = m_shared.to(device), pres_shared.to(device)
            m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, super_list)
            logits_y, logits_i = model(x, m_ind)
            corr += (logits_y.argmax(1) == y).sum().item(); tot += y.size(0)
            f1s.append(presence_f1_from_logits(logits_i, y_ind))
    return corr/max(1,tot), float(np.mean(f1s))

## Base ResNet

In [43]:
import os, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score
import numpy as np

In [44]:
class PartImageNetDatasetBS(Dataset):
    def __init__(self, root, ann_path, transform=None):
        self.root = root
        self.transform = transform
        with open(ann_path, "r") as f:
            self.ann = json.load(f)

        self.images = {}
        for obj in self.ann["images"]:
            self.images[obj["id"]] = obj["file_name"]

        self.parts = {}
        for ann in self.ann["annotations"]:
            img_id = ann["image_id"]
            cat_id = ann["category_id"]
            if img_id not in self.parts:
                self.parts[img_id] = []
            self.parts[img_id].append(cat_id)

        self.categories = self.ann["categories"]
        self.supercats = sorted(set([c["supercategory"] for c in self.categories]))
        self.super_to_idx = {s:i for i,s in enumerate(self.supercats)}

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_id = list(self.images.keys())[idx]
        img_path = os.path.join(self.root, self.images[img_id])
        image = Image.open(img_path).convert("RGB")

        label = self.super_to_idx[self.categories[self.parts[img_id][0]]["supercategory"]] \
                if img_id in self.parts else 0

        # Multi-hot part vector
        part_vec = np.zeros(len(self.categories), dtype=np.float32)
        if img_id in self.parts:
            for cid in self.parts[img_id]:
                part_vec[cid] = 1.0

        if self.transform:
            image = self.transform(image)

        return image, label, torch.tensor(part_vec)

In [45]:
class BaselineMTLResNet(nn.Module):
    def __init__(self, num_labels, num_parts):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc_label = nn.Linear(base.fc.in_features, num_labels)
        self.fc_part = nn.Linear(base.fc.in_features, num_parts)

    def forward(self, x):
        feats = self.backbone(x).flatten(1)
        out_label = self.fc_label(feats)
        out_part = torch.sigmoid(self.fc_part(feats))
        return out_label, out_part
    
def presence_f1(y_true, y_pred):
    y_true = y_true.cpu().numpy()
    y_pred = (y_pred.cpu().numpy() > 0.5).astype(int)
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

In [46]:
def train_one_epoch(model, loader, opt, criterion_c, criterion_p):
    model.train()
    total, correct, total_loss = 0,0,0
    for imgs, labels, parts in loader:
        imgs, labels, parts = imgs.to(device), labels.to(device), parts.to(device)
        opt.zero_grad()
        out_c, out_p = model(imgs)
        loss_c = criterion_c(out_c, labels)
        loss_p = criterion_p(out_p, parts)
        loss = loss_c + 0.5*loss_p
        loss.backward()
        opt.step()
        total_loss += loss.item()
        total += labels.size(0)
        correct += (out_c.argmax(1) == labels).sum().item()
    return correct/total, total_loss/len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion_c, criterion_p):
    model.eval()
    total, correct, total_loss = 0,0,0
    all_true, all_pred = [], []
    for imgs, labels, parts in loader:
        imgs, labels, parts = imgs.to(device), labels.to(device), parts.to(device)
        out_c, out_p = model(imgs)
        loss_c = criterion_c(out_c, labels)
        loss_p = criterion_p(out_p, parts)
        loss = loss_c + 0.5*loss_p
        total_loss += loss.item()
        total += labels.size(0)
        correct += (out_c.argmax(1) == labels).sum().item()
        all_true.append(parts.cpu())
        all_pred.append(out_p.cpu())
    f1 = presence_f1(torch.cat(all_true), torch.cat(all_pred))
    return correct/total, total_loss/len(loader), f1

## Integrated Training

In [47]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from typing import List, Tuple, Dict, Optional
import copy

In [48]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

In [49]:
# Replace your current collate_fn with this one:
def collate_fn(batch):
    import torch
    good = []
    for item in batch:
        # each item should be: (img, label_idx, mask[C,H,W], presence[C], haspix_flag)
        if item is None: 
            continue
        if not isinstance(item, (tuple, list)) or len(item) != 5:
            continue
        x, y, m, pres, haspix = item
        if m is None or pres is None:
            continue
        # Per-item checks: mask is 3D [C,H,W], presence is 1D [C]
        if not hasattr(m, "ndim") or m.ndim != 3:
            continue
        if not hasattr(pres, "ndim") or pres.ndim != 1:
            continue
        if m.shape[0] != pres.shape[0]:
            continue
        good.append((x, y, m, pres, haspix))

    if not good:
        raise ValueError("All items in batch were invalid; let DataLoader resample.")

    xs   = torch.stack([g[0] for g in good], dim=0)                       # [B,3,H,W]
    ys   = torch.tensor([int(g[1]) for g in good], dtype=torch.long)      # [B]
    ms   = torch.stack([g[2] for g in good], dim=0)                       # [B,C,H,W] (C can be 0)
    pres = torch.stack([g[3] for g in good], dim=0).to(torch.float32)     # [B,C]
    hpix = torch.tensor([float(g[4]) for g in good], dtype=torch.float32) # [B]
    return xs, ys, ms, pres, hpix


In [50]:
@torch.no_grad()
def macro_presence_f1_from_logits_shared(logits_shared: torch.Tensor,
                                         pres_shared: torch.Tensor,
                                         thresh: float = 0.5,
                                         eps: float = 1e-6) -> float:
    """
    logits_shared: (B, P_shared) raw logits for shared parts
    pres_shared:   (B, P_shared) in {0,1}
    """
    prob = torch.sigmoid(logits_shared)
    pred = (prob >= thresh).float()

    # Per-channel F1, then macro
    tp = (pred * pres_shared).sum(dim=0)
    fp = (pred * (1 - pres_shared)).sum(dim=0)
    fn = ((1 - pred) * pres_shared).sum(dim=0)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    return float(f1.mean().item())

@torch.no_grad()
def acc_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    return float((logits.argmax(1) == y).float().mean().item())


In [51]:
@torch.no_grad()
def _set_bn_eval(m):
    if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
        m.eval()

@torch.no_grad()
def _macro_f1_from_logits(L, Y, thr=0.5, eps=1e-6):
    P = L.sigmoid()
    pred = (P >= thr).float()
    tp = (pred*Y).sum(0); fp = (pred*(1-Y)).sum(0); fn = ((1-pred)*Y).sum(0)
    f1 = (2*tp)/(2*tp + fp + fn + eps)
    return float(f1.mean().item())

In [52]:
@dataclass
class IndepSpec:
    supers: List[str]
    parts: List[str]                               # == PART_VOCAB
    tuples: List[Tuple[str,str]]                   # [(super, part), ...]
    tuple_to_idx: Dict[Tuple[str,str], int]
    part_to_shared: Dict[str, int]                 # part name -> shared idx

def build_indep_spec(supercats: List[str]) -> IndepSpec:
    tuples: List[Tuple[str,str]] = []
    for s in supercats:
        for p in sorted(SUPER_TO_PARTS.get(s, set())):
            tuples.append((s, p))
    tuple_to_idx = {t:i for i,t in enumerate(tuples)}
    part_to_shared = {p: PART_TO_IDX[p] for p in PART_VOCAB}
    return IndepSpec(
        supers=list(supercats),
        parts=list(PART_VOCAB),
        tuples=tuples,
        tuple_to_idx=tuple_to_idx,
        part_to_shared=part_to_shared,
    )

def pack_shared_to_indep_batch(m_shared: torch.Tensor, pres_shared: torch.Tensor,
                               super_idx: torch.Tensor, spec: IndepSpec,
                               super_list: List[str]):
    """
    Convert batch of shared masks/presence to independent (super,part) layout.
    Inputs:
      - m_shared:   (B,P,H,W)
      - pres_shared:(B,P)
      - super_idx:  (B,) indices into super_list
    Returns:
      - m_ind: (B,K,H,W) aligned to spec.tuples
      - y_ind: (B,K)     presence aligned to spec.tuples
    Fills zeros for tuples not applicable to a sample’s superclass.
    """
    B, P, H, W = m_shared.shape
    K = len(spec.tuples)
    m_ind = torch.zeros(B, K, H, W, device=m_shared.device, dtype=m_shared.dtype)
    y_ind = torch.zeros(B, K, device=pres_shared.device, dtype=pres_shared.dtype)
    for b in range(B):
        sname = super_list[int(super_idx[b].item())]
        for p in SUPER_TO_PARTS.get(sname, set()):
            k  = spec.tuple_to_idx[(sname, p)]
            ps = spec.part_to_shared[p]
            m_ind[b, k] = m_shared[b, ps]
            y_ind[b, k] = pres_shared[b, ps]
    return m_ind, y_ind

In [53]:
@torch.no_grad()
def indep_logits_to_shared(logits_ind: torch.Tensor, y: torch.Tensor,
                           spec: IndepSpec, super_list: List[str]) -> torch.Tensor:
    """For each sample b, pick the tuples of its supercategory and map to shared parts.
    Returns (B, P_shared) logits (fill -inf for parts not in that super)."""
    B = logits_ind.size(0)
    P = len(PART_VOCAB)
    out = torch.full((B, P), fill_value=-1e4, device=logits_ind.device)
    for b in range(B):
        sname = super_list[int(y[b].item())]
        for p in SUPER_TO_PARTS.get(sname, set()):
            k = spec.tuple_to_idx[(sname, p)]
            out[b, PART_TO_IDX[p]] = logits_ind[b, k]
    return out

def pack_shared_to_indep_batch(m_shared: torch.Tensor, pres_shared: torch.Tensor,
                               super_idx: torch.Tensor, spec: IndepSpec,
                               super_list: List[str]):
    """
    Convert batch of shared masks/presence to independent (super,part) layout.
    Inputs:
      - m_shared:   (B,P,H,W)
      - pres_shared:(B,P)
      - super_idx:  (B,) indices into super_list
    Returns:
      - m_ind: (B,K,H,W) aligned to spec.tuples
      - y_ind: (B,K)     presence aligned to spec.tuples
    Fills zeros for tuples not applicable to a sample’s superclass.
    """
    B, P, H, W = m_shared.shape
    K = len(spec.tuples)
    m_ind = torch.zeros(B, K, H, W, device=m_shared.device, dtype=m_shared.dtype)
    y_ind = torch.zeros(B, K, device=pres_shared.device, dtype=pres_shared.dtype)
    for b in range(B):
        sname = super_list[int(super_idx[b].item())]
        for p in SUPER_TO_PARTS.get(sname, set()):
            k  = spec.tuple_to_idx[(sname, p)]
            ps = spec.part_to_shared[p]
            m_ind[b, k] = m_shared[b, ps]
            y_ind[b, k] = pres_shared[b, ps]
    return m_ind, y_ind


In [ ]:
def get_model(model_type, num_classes, num_parts):
    if model_type == "shared":
        return SharedMTLResNet(num_classes=num_classes, num_parts=num_parts)
    elif model_type == "independent":
        return IndependentMTLResNet(num_classes=num_classes, num_parts=num_parts)
    elif model_type == "baseline":
        return BaselineMTLResNet(num_classes=num_classes, num_parts=num_parts)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

In [ ]:
def run_baseline(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    num_parts  = len(PART_VOCAB)
    model = BaselineMTLResNet(num_labels, num_parts).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, _m, pres, _ in tl:
            x, y, pres = x.to(device), y.to(device), pres.to(device)
            logits_y, logits_p = model(x)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_p, pres)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, _m, pres, _ in vl:
                x, y, pres = x.to(device), y.to(device), pres.to(device)
                logits_y, logits_p = model(x)
                accs.append(_acc_from_logits(logits_y, y))
                f1s.append(presence_f1(torch.sigmoid(logits_p), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[BASE]  Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

def run_shared(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    num_parts  = len(PART_VOCAB)
    model = SharedMTLResNet(num_labels, num_parts).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, m, pres, _ in tl:
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            logits_y, logits_p = model(x, m)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_p, pres)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, m, pres, _ in vl:
                x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
                logits_y, logits_p = model(x, m)
                accs.append(_acc_from_logits(logits_y, y))
                f1s.append(presence_f1(torch.sigmoid(logits_p), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[SHARED] Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

def run_indep(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    spec = build_indep_spec(train_ds.supercats)
    K = len(spec.tuples)
    model = IndependentMTLResNet(num_labels, K).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, m, pres, _ in tl:
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            m_ind, y_ind = pack_shared_to_indep_batch(m, pres, y, spec, train_ds.supercats)
            logits_y, logits_i = model(x, m_ind)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_i, y_ind)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval: map indep part logits back to SHARED space for presence-F1
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, m, pres, _ in vl:
                x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
                m_ind, _ = pack_shared_to_indep_batch(m, pres, y, spec, val_ds.supercats)
                logits_y, logits_i = model(x, m_ind)
                accs.append(_acc_from_logits(logits_y, y))
                logits_shared = indep_logits_to_shared(logits_i, y, spec, val_ds.supercats)
                f1s.append(presence_f1(torch.sigmoid(logits_shared), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[INDEP]  Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

In [ ]:
def compare_all(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    import pandas as pd

    print("=== Running Baseline (RGB-only) ===")
    res_base   = run_baseline(epochs, lr, bs, train_loader, val_loader)

    print("\n=== Running Shared-channels ===")
    res_shared = run_shared(epochs, lr, bs, train_loader, val_loader)

    print("\n=== Running Independent-channels ===")
    res_indep  = run_indep(epochs, lr, bs, train_loader, val_loader)

    summary = pd.DataFrame([
        {"Method": "Baseline (RGB)",      "Best Label Acc": res_base['acc'],   "Best Part F1": res_base['f1']},
        {"Method": "Shared-channels",      "Best Label Acc": res_shared['acc'], "Best Part F1": res_shared['f1']},
        {"Method": "Independent-channels", "Best Label Acc": res_indep['acc'],  "Best Part F1": res_indep['f1']},
    ])
    print("\n=== Summary ===")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return summary

In [ ]:
@torch.no_grad()
def evaluate_shared(model, loader, device, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, m, pres, _ in loader:
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device).float()
        logits_y, logits_p = model(x, m)

        # --- accuracy ---
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()

        # --- presence F1 collection ---
        # strong guards (catch the kinds of bugs that produced 0.26 before)
        assert logits_p.ndim == 2, f"parts logits must be (B,C); got {logits_p.shape}"
        assert pres.ndim     == 2, f"presence must be (B,C); got {pres.shape}"
        assert logits_p.shape == pres.shape, f"shape mismatch {logits_p.shape} vs {pres.shape}"
        L_parts.append(logits_p.detach().cpu())
        Y_parts.append(pres.detach().cpu())

    acc = correct / max(total, 1)
    L = torch.cat(L_parts) if L_parts else torch.empty(0, device='cpu')
    Y = torch.cat(Y_parts) if Y_parts else torch.empty(0, device='cpu')
    f1 = _macro_f1_from_logits(L, Y, thr=thr) if L.numel() else float("nan")
    return acc, f1

@torch.no_grad()
def evaluate_baseline(model, loader, device, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, _m, pres, _ in loader:
        x, y, pres = x.to(device), y.to(device), pres.to(device).float()
        logits_y, logits_p = model(x)
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()
        assert logits_p.ndim == 2 and pres.ndim == 2 and logits_p.shape == pres.shape
        L_parts.append(logits_p.detach().cpu())
        Y_parts.append(pres.detach().cpu())
    acc = correct / max(total, 1)
    L = torch.cat(L_parts); Y = torch.cat(Y_parts)
    f1 = _macro_f1_from_logits(L, Y, thr=thr)
    return acc, f1

@torch.no_grad()
def evaluate_indep(model, loader, device, spec, super_list, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, m, pres, _ in loader:
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device).float()
        m_ind, y_ind = pack_shared_to_indep_batch(m, pres, y, spec, super_list)
        logits_y, logits_i = model(x, m_ind)
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()
        logits_shared = indep_logits_to_shared(logits_i, y, spec, super_list)
        assert logits_shared.shape == pres.shape, f"indep→shared shape mismatch {logits_shared.shape} vs {pres.shape}"
        L_parts.append(logits_shared.detach().cpu())
        Y_parts.append(pres.detach().cpu())
    acc = correct / max(total, 1)
    L = torch.cat(L_parts); Y = torch.cat(Y_parts)
    f1 = _macro_f1_from_logits(L, Y, thr=thr)
    return acc, f1

In [ ]:
class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0, mode="max", monitor="val_acc"):
        assert mode in {"max", "min"}
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.monitor = monitor
        self.best = -float("inf") if mode == "max" else float("inf")
        self.num_bad = 0
        self.stopped = False
        self.stopped_epoch = None

    def _is_improvement(self, value):
        if self.mode == "max":
            return value >= self.best + self.min_delta
        else:
            return value <= self.best - self.min_delta

    def step(self, metric_value, epoch):
        """Returns (should_stop, is_new_best)."""
        is_best = False
        if self._is_improvement(metric_value):
            self.best = metric_value
            self.num_bad = 0
            is_best = True
        else:
            self.num_bad += 1
            if self.num_bad >= self.patience:
                self.stopped = True
                self.stopped_epoch = epoch
        return self.stopped, is_best

In [ ]:
def train_all_together(
    epochs=3,
    lr=1e-3,
    bs=32,
    alpha_part=0.1,         # weight for part presence BCE
    train_loader=None,
    val_loader=None,
    device=None
):
    dev = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build dataloaders if not provided (assumes train_ds / val_ds / collate_fn exist in the notebook)
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

    # Model setup
    num_labels = len(train_ds.supercats)
    num_shared_parts = len(PART_VOCAB)

    baseline = BaselineMTLResNet(num_labels, num_shared_parts).to(dev)
    shared   = SharedMTLResNet(num_labels, num_shared_parts).to(dev)

    # Independent spec based on supercats present in training set
    spec = build_indep_spec(train_ds.supercats)
    num_indep_parts = len(spec.tuples)
    indep   = IndependentMTLResNet(num_labels, num_indep_parts).to(dev)

    opt_base   = torch.optim.Adam(baseline.parameters(), lr=lr)
    opt_shared = torch.optim.Adam(shared.parameters(),   lr=lr)
    opt_indep  = torch.optim.Adam(indep.parameters(),    lr=lr)

    # Logging
    rows = []

    for ep in range(1, epochs + 1):
        baseline.train(); shared.train(); indep.train()

        # Running stats (train)
        tr_acc_base, tr_f1_base = [], []
        tr_acc_shared, tr_f1_shared = [], []
        tr_acc_indep, tr_f1_indep = [], []

        for step, (x, y, m_shared, pres_shared, _haspix) in enumerate(tl, start=1):
            x, y = x.to(dev), y.to(dev)
            m_shared, pres_shared = m_shared.to(dev), pres_shared.to(dev)

            # ----- Baseline (RGB only) -----
            logits_y_b, logits_p_b = baseline(x)
            loss_b = F.cross_entropy(logits_y_b, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_b, pres_shared)
            opt_base.zero_grad()
            loss_b.backward()
            opt_base.step()

            tr_acc_base.append(acc_from_logits(logits_y_b, y))
            tr_f1_base.append(macro_presence_f1_from_logits_shared(logits_p_b, pres_shared))

            # ----- Shared-channels (RGB + shared masks) -----
            logits_y_s, logits_p_s = shared(x, m_shared)
            loss_s = F.cross_entropy(logits_y_s, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_s, pres_shared)
            opt_shared.zero_grad()
            loss_s.backward()
            opt_shared.step()

            tr_acc_shared.append(acc_from_logits(logits_y_s, y))
            tr_f1_shared.append(macro_presence_f1_from_logits_shared(logits_p_s, pres_shared))

            # ----- Independent-channels (RGB + (super, part) channels) -----
            # pack the current batch’s shared masks/presence -> independent layout for each sample’s supercategory
            m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, train_ds.supercats)
            logits_y_i, logits_i = indep(x, m_ind)
            loss_i = F.cross_entropy(logits_y_i, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_i, y_ind)
            opt_indep.zero_grad()
            loss_i.backward()
            opt_indep.step()

            # For F1 we compare in SHARED space for apples-to-apples
            logits_shared_i = indep_logits_to_shared(logits_i, y, spec, train_ds.supercats)
            tr_acc_indep.append(acc_from_logits(logits_y_i, y))
            tr_f1_indep.append(macro_presence_f1_from_logits_shared(logits_shared_i, pres_shared))

            # (Optional) occasional progress print
            if step % 100 == 0:
                print(f"Epoch {ep:02d} | Step {step:04d} "
                      f"| base acc {np.mean(tr_acc_base):.3f} shared acc {np.mean(tr_acc_shared):.3f} indep acc {np.mean(tr_acc_indep):.3f}")

        # ---- End of epoch: evaluate on val ----
        va_acc_base, va_f1_base     = evaluate_baseline(baseline, vl, dev)
        va_acc_shared, va_f1_shared = evaluate_shared(shared, vl, dev)
        va_acc_indep, va_f1_indep   = evaluate_indep(indep, vl, dev, spec, val_ds.supercats)

        # Aggregate train stats (averages across batches)
        tr_acc_base_m,   tr_f1_base_m   = float(np.mean(tr_acc_base)),   float(np.mean(tr_f1_base))
        tr_acc_shared_m, tr_f1_shared_m = float(np.mean(tr_acc_shared)), float(np.mean(tr_f1_shared))
        tr_acc_indep_m,  tr_f1_indep_m  = float(np.mean(tr_acc_indep)),  float(np.mean(tr_f1_indep))

        print(f"[E{ep:02d}] BASE:   train acc {tr_acc_base_m:6.2%} | val acc {va_acc_base:6.2%} | train F1 {tr_f1_base_m:.3f} | val F1 {va_f1_base:.3f}")
        print(f"[E{ep:02d}] SHARED: train acc {tr_acc_shared_m:6.2%} | val acc {va_acc_shared:6.2%} | train F1 {tr_f1_shared_m:.3f} | val F1 {va_f1_shared:.3f}")
        print(f"[E{ep:02d}] INDEP:  train acc {tr_acc_indep_m:6.2%} | val acc {va_acc_indep:6.2%} | train F1 {tr_f1_indep_m:.3f} | val F1 {va_f1_indep:.3f}")

        rows.append({
            "epoch": ep,

            "BASE_train_acc":   tr_acc_base_m,
            "BASE_val_acc":     va_acc_base,
            "BASE_train_f1":    tr_f1_base_m,
            "BASE_val_f1":      va_f1_base,

            "SHARED_train_acc": tr_acc_shared_m,
            "SHARED_val_acc":   va_acc_shared,
            "SHARED_train_f1":  tr_f1_shared_m,
            "SHARED_val_f1":    va_f1_shared,

            "INDEP_train_acc":  tr_acc_indep_m,
            "INDEP_val_acc":    va_acc_indep,
            "INDEP_train_f1":   tr_f1_indep_m,
            "INDEP_val_f1":     va_f1_indep,
        })

    results = pd.DataFrame(rows)
    display(results)

    # Quick summary of best validation performance per model
    best_summary = pd.DataFrame([
        {
            "Model": "Baseline (RGB)",
            "Best Val Acc": results["BASE_val_acc"].max(),
            "Best Val F1":  results["BASE_val_f1"].max(),
        },
        {
            "Model": "Shared-channels",
            "Best Val Acc": results["SHARED_val_acc"].max(),
            "Best Val F1":  results["SHARED_val_f1"].max(),
        },
        {
            "Model": "Independent-channels",
            "Best Val Acc": results["INDEP_val_acc"].max(),
            "Best Val F1":  results["INDEP_val_f1"].max(),
        },
    ])
    display(best_summary)

    return {
        "history": results,
        "best": best_summary,
        "models": {"baseline": baseline, "shared": shared, "indep": indep},
        "spec": spec,
    }

In [ ]:
ROOT = "PartImageNet_Seg/PartImageNet"

TRAIN_JSON     = os.path.join(ROOT, "annotations/train/train.json")
VAL_JSON       = os.path.join(ROOT, "annotations/val/val.json")
TRAIN_IMG_ROOT = os.path.join(ROOT, "images/train")
VAL_IMG_ROOT   = os.path.join(ROOT, "images/val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = PartImageNetCOCODataset(
    ann_path=TRAIN_JSON,
    img_root=TRAIN_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=True,
    shared_by_type=True,
)
val_ds = PartImageNetCOCODataset(
    ann_path=VAL_JSON,
    img_root=VAL_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=False,
    shared_by_type=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          collate_fn=collate_fn, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          collate_fn=collate_fn, drop_last=False)

print("\nRunning audit on training data (first 200 images)…")
audit_masks(train_ds, max_images=200)

In [ ]:
out = train_all_together(epochs=10, lr=1e-3, bs=32, train_loader=train_loader, val_loader=val_loader, device=device)

In [ ]:
out = train_all_together(epochs=10, lr=1e-3, bs=32, train_loader=train_loader, val_loader=val_loader, device=device)

Epoch 01 | Step 0100 | base acc 0.464 shared acc 0.311 indep acc 0.294
Epoch 01 | Step 0200 | base acc 0.486 shared acc 0.324 indep acc 0.303
Epoch 01 | Step 0300 | base acc 0.509 shared acc 0.339 indep acc 0.312
Epoch 01 | Step 0400 | base acc 0.531 shared acc 0.351 indep acc 0.331
Epoch 01 | Step 0500 | base acc 0.545 shared acc 0.360 indep acc 0.355
Epoch 01 | Step 0600 | base acc 0.557 shared acc 0.362 indep acc 0.380
Epoch 01 | Step 0700 | base acc 0.565 shared acc 0.370 indep acc 0.405
Epoch 01 | Step 0800 | base acc 0.571 shared acc 0.379 indep acc 0.440
Epoch 01 | Step 0900 | base acc 0.576 shared acc 0.384 indep acc 0.468
Epoch 01 | Step 1000 | base acc 0.584 shared acc 0.391 indep acc 0.496
Epoch 01 | Step 1100 | base acc 0.590 shared acc 0.399 indep acc 0.522
Epoch 01 | Step 1200 | base acc 0.597 shared acc 0.407 indep acc 0.545
[E01] BASE:   train acc 60.10% | val acc 64.43% | train F1 0.290 | val F1 0.297
[E01] SHARED: train acc 41.32% | val acc 46.93% | train F1 0.273 | v

,epoch,BASE_train_acc,BASE_val_acc,BASE_train_f1,BASE_val_f1,SHARED_train_acc,SHARED_val_acc,SHARED_train_f1,SHARED_val_f1,INDEP_train_acc,INDEP_val_acc,INDEP_train_f1,INDEP_val_f1
0,1,0.601028,0.644279,0.290396,0.297249,0.413216,0.469320,0.272772,0.533192,0.562184,0.800995,0.073502,0.222921
1,2,0.709472,0.543947,0.290401,0.297249,0.647567,0.722222,0.418428,0.622971,0.858886,0.736318,0.186288,0.234863
2,3,0.746726,0.744610,0.290163,0.297249,0.768230,0.711443,0.489652,0.740599,0.912421,0.935323,0.206122,0.339200
3,4,0.775283,0.747927,0.290094,0.297249,0.816273,0.543947,0.526443,0.445259,0.925608,0.873964,0.211063,0.280392
4,5,0.800016,0.767828,0.290241,0.297249,0.844832,0.805141,0.550785,0.757994,0.929853,0.769486,0.214945,0.256041
5,6,0.820397,0.786899,0.290171,0.297249,0.851205,0.868159,0.556202,0.873490,0.939083,0.919569,0.218901,0.316700
6,7,0.835939,0.784411,0.290325,0.297249,0.871816,0.748756,0.565050,0.802835,0.949158,0.939469,0.220804,0.350535
7,8,0.854316,0.781924,0.290234,0.297249,0.872619,0.451907,0.567677,0.573353,0.945345,0.902156,0.221376,0.285070
8,9,0.862581,0.800995,0.290301,0.297249,0.885051,0.669154,0.568235,0.632227,0.948683,0.621891,0.222498,0.149528
9,10,0.879199,0.785240,0.290212,0.297249,0.898396,0.637645,0.574525,0.614781,0.948026,0.947761,0.221978,0.328348


,Model,Best Val Acc,Best Val F1
0,Baseline (RGB),0.800995,0.297249
1,Shared-channels,0.868159,0.873490
2,Independent-channels,0.947761,0.350535


In [ ]:
import torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VAL_BS = 64
val_loader = DataLoader(val_ds, batch_size=VAL_BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
train_loader_small = DataLoader(train_ds, batch_size=VAL_BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)

def to_cpu(x): 
    return x.detach().float().cpu()

@torch.no_grad()
def batch_f1_at_threshold(logits, labels, thr=0.5, eps=1e-6):
    """Macro F1 over parts for a batch."""
    probs = logits.sigmoid()
    pred = (probs >= thr).float()
    tp = (pred*labels).sum(0); fp = (pred*(1-labels)).sum(0); fn = ((1-pred)*labels).sum(0)
    f1 = (2*tp)/(2*tp+fp+fn+eps)
    return float(f1.mean())

def get_shared_logits_labels(model, loader, max_batches=None):
    """Collect logits and labels for the SHARED head over a loader."""
    model.eval()
    all_logits, all_labels = []
    for bi,(x,y,m,pres,_) in enumerate(loader):
        x, m, pres = x.to(device), m.to(device), pres.to(device).float()
        _, lp = model(x, m)
        all_logits.append(to_cpu(lp))
        all_labels.append(to_cpu(pres))
        if max_batches is not None and (bi+1)>=max_batches:
            break
    return torch.cat(all_logits), torch.cat(all_labels)


In [ ]:
print("train parts:", train_ds.part_types)
print("val   parts:", val_ds.part_types)
print("train C:", train_ds.num_part_channels, " | val C:", val_ds.num_part_channels)

same_order = (train_ds.part_types == val_ds.part_types)
print("Same list & order?", same_order)


train parts: ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
val   parts: ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
train C: 13  | val C: 13
Same list & order? True


In [ ]:
# Compare part-type lists between splits
import hashlib, json
def md5_list(lst): return hashlib.md5(json.dumps(lst).encode()).hexdigest()

print("train part_types hash:", md5_list(train_ds.part_types), "len:", len(train_ds.part_types))
print("val   part_types hash:", md5_list(val_ds.part_types),   "len:", len(val_ds.part_types))


train part_types hash: d64c00eb5fb80fe3d3626ba33efc4d57 len: 13
val   part_types hash: d64c00eb5fb80fe3d3626ba33efc4d57 len: 13


In [ ]:
import torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Utilities ----
@torch.no_grad()
def collect_logits_labels_shared(model, loader, max_batches=None):
    model.eval()
    all_logits, all_labels = [], []
    for bi,(x,y,m,pres,_) in enumerate(loader):
        x, m, pres = x.to(device), m.to(device), pres.to(device).float()
        _, lp = model(x, m)
        all_logits.append(lp.detach().cpu())
        all_labels.append(pres.detach().cpu())
        if max_batches is not None and (bi+1)>=max_batches:
            break
    return torch.cat(all_logits), torch.cat(all_labels)

@torch.no_grad()
def macro_f1_at_threshold(logits, labels, thr=0.5, eps=1e-6, mask=None):
    probs = logits.sigmoid()
    pred  = (probs >= thr).float()
    if mask is not None:
        labels = labels[:, mask]
        pred   = pred[:, mask]
    tp = (pred*labels).sum(0); fp = (pred*(1-labels)).sum(0); fn = ((1-pred)*labels).sum(0)
    f1 = (2*tp)/(2*tp+fp+fn+eps)
    return float(f1.mean())

@torch.no_grad()
def per_part_stats(logits, labels, thr=0.5, eps=1e-6):
    probs = logits.sigmoid()
    pred  = (probs >= thr).float()
    tp = (pred*labels).sum(0); fp = (pred*(1-labels)).sum(0); fn = ((1-pred)*labels).sum(0)
    prec = tp/(tp+fp+eps); rec = tp/(tp+fn+eps); f1 = 2*prec*rec/(prec+rec+eps)
    support = labels.sum(0)
    prate = labels.mean(0)
    names = getattr(train_ds, "part_types", [f"p{j}" for j in range(labels.shape[1])])
    return pd.DataFrame({
        "part": names,
        "support_val": support.numpy(),
        "presence_rate_val": prate.numpy(),
        "precision": prec.numpy(),
        "recall": rec.numpy(),
        "f1": f1.numpy()
    }).sort_values("f1")

def presence_support(loader, max_batches=None):
    tot = None; n=0
    for bi,(_,_,_,pres,_) in enumerate(loader):
        pres = pres.float()
        tot = pres.sum(0) if tot is None else tot + pres.sum(0)
        n += pres.shape[0]
        if max_batches is not None and (bi+1)>=max_batches: break
    return tot.cpu().numpy(), n

def threshold_sweep(logits, labels, mask=None, grid=None):
    if grid is None: grid = torch.linspace(0.05, 0.95, steps=19)
    f1s = [macro_f1_at_threshold(logits, labels, float(t), mask=mask) for t in grid]
    best_idx = int(np.argmax(f1s))
    return float(grid[best_idx]), float(f1s[best_idx]), float(f1s[int((len(grid)-1)/2)])  # best thr, best F1, F1@0.5

# ---- 1) Check label regime & BN/dropout behavior ----
def audit_regime_and_bn(shared_model, train_loader, val_loader):
    # Fraction of images that actually contain any part pixels
    haspix_train = []
    for _,_,_,pres,_ in train_loader:
        haspix_train.append((pres.sum(1) > 0).float().mean().item()); break
    haspix_val = []
    for _,_,_,pres,_ in val_loader:
        haspix_val.append((pres.sum(1) > 0).float().mean().item()); break
    print(f"[has-pixels] train batch frac≈{np.mean(haspix_train):.3f} | val batch frac≈{np.mean(haspix_val):.3f}")

    # Make sure we're in eval and optionally freeze BN to avoid small-batch drift
    shared_model.eval()
    def set_bn_eval(m):
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm): m.eval()
    shared_model.apply(set_bn_eval)

# ---- 2) Run the core diagnostics ----
def run_debug_val_f1(shared_model, train_loader, val_loader, min_support=1):
    audit_regime_and_bn(shared_model, train_loader, val_loader)

    # Collect full val logits/labels
    L_val, Y_val = collect_logits_labels_shared(shared_model, val_loader)
    # Also a small sample of train for calibration
    L_tr,  Y_tr  = collect_logits_labels_shared(shared_model, train_loader, max_batches=128)

    # Support mask: only parts with at least `min_support` positives in VAL
    sup_counts, _ = presence_support(val_loader)
    support_mask = (sup_counts >= min_support)

    print(f"[support] parts with ≥{min_support} positives in VAL: {int(support_mask.sum())}/{len(support_mask)}")

    # F1 @ 0.5 (all vs supported)
    f1_all_05 = macro_f1_at_threshold(L_val, Y_val, 0.5, mask=None)
    f1_sup_05 = macro_f1_at_threshold(L_val, Y_val, 0.5, mask=support_mask)
    print(f"[macro-F1] VAL @0.5 -> all parts: {f1_all_05:.3f} | supported-only: {f1_sup_05:.3f}")

    # Threshold sweep on VAL (diagnostic only)
    best_thr_val, best_f1_val, f1_val_05 = threshold_sweep(L_val, Y_val, mask=support_mask)
    print(f"[thr sweep] VAL best thr={best_thr_val:.2f} | best macro-F1={best_f1_val:.3f} | @0.5={f1_val_05:.3f} (supported)")

    # Thresholds fitted on TRAIN (use these for fair eval)
    best_thr_tr, best_f1_tr_est, f1_tr_05 = threshold_sweep(L_tr, Y_tr, mask=None)
    print(f"[thr sweep] TRAIN best thr={best_thr_tr:.2f} (for calibration) | est train macro-F1={best_f1_tr_est:.3f}")

    # Apply train-calibrated threshold to VAL
    f1_val_with_trainthr_all = macro_f1_at_threshold(L_val, Y_val, best_thr_tr, mask=None)
    f1_val_with_trainthr_sup = macro_f1_at_threshold(L_val, Y_val, best_thr_tr, mask=support_mask)
    print(f"[calibrated] VAL macro-F1 with train thr -> all: {f1_val_with_trainthr_all:.3f} | supported: {f1_val_with_trainthr_sup:.3f}")

    # Per-part breakdown at 0.5 (to see precision vs recall failure mode)
    print("\nWorst 15 parts (VAL, @0.5):")
    display(per_part_stats(L_val, Y_val, thr=0.5).head(15))

    return {
        "L_val": L_val, "Y_val": Y_val,
        "support_mask": support_mask,
        "best_thr_train": best_thr_tr
    }


In [ ]:
VAL_BS = 64
train_loader_dbg = DataLoader(train_ds, batch_size=VAL_BS, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader_dbg   = DataLoader(val_ds,   batch_size=VAL_BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

shared = out["models"]["shared"]

diag = run_debug_val_f1(shared, train_loader_dbg, val_loader_dbg, min_support=3)


NameError: name 'DataLoader' is not defined

In [ ]:
def compare_all(epochs: int = 3, lr: float = 1e-3, bs: int = 32, train_loader=None, val_loader=None):
    print("=== Running Baseline (RGB-only) ===")
    res_base   = run_baseline(epochs, lr, bs, train_loader, val_loader)
    print("=== Running Shared-channels ===")
    res_shared = run_shared(epochs, lr, bs, train_loader, val_loader)
    print("=== Running Independent-channels ===")
    res_indep  = run_indep(epochs, lr, bs, train_loader, val_loader)

    # Summarize
    import pandas as pd
    summary = pd.DataFrame([
        {"Method": "Baseline (RGB)",      "Best Label Acc": res_base['acc'],   "Best Part F1": res_base['f1']},
        {"Method": "Shared-channels",      "Best Label Acc": res_shared['acc'], "Best Part F1": res_shared['f1']},
        {"Method": "Independent-channels", "Best Label Acc": res_indep['acc'],  "Best Part F1": res_indep['f1']},
    ])
    print("=== Summary ===")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return summary

In [ ]:
summary_df = compare_all(epochs=3, lr=1e-3, bs=32)

=== Running Baseline (RGB-only) ===


NameError: name '_acc_from_logits' is not defined

In [ ]:
# Independent Channels Training
specs = build_indep_spec(train_ds.supercats)
num_parts  = train_ds.num_part_channels
K = len(specs.tuples)

# Safely free any previous model
for k in list(globals().keys()):
    if k == 'model':
        del globals()[k]
torch.cuda.empty_cache()

indep_model = IndependentMTLResNet(num_labels=len(train_ds.supercats), num_indep_parts=K).to(device)
indep_opt = torch.optim.Adam(indep_model.parameters(), lr=1e-3)

EPOCHS = 10
stopper = EarlyStopper(patience=3, min_delta=0.0)

for epoch in range(1, EPOCHS+1):
    tr_acc, tr_f1 = train_one_epoch_indep(indep_model, train_loader, specs, train_ds.supercats, indep_opt)
    va_acc, va_f1 = evaluate_indep(indep_model, val_loader, specs, val_ds.supercats)

    print(f"Epoch {epoch:02d} | Train Acc {tr_acc*100:5.2f}% | Val Acc {va_acc*100:5.2f}% | Part Presence F1 {va_f1:.3f}")

    stopper.step(va_acc)
    if stopper.stop:
        print("Early stopped.")
        break

[indep] step 0 | zero indep-channels across batch: 32/42
[indep] step 100 | zero indep-channels across batch: 28/42
[indep] step 200 | zero indep-channels across batch: 32/42
[indep] step 300 | zero indep-channels across batch: 34/42
[indep] step 400 | zero indep-channels across batch: 36/42
[indep] step 500 | zero indep-channels across batch: 34/42
[indep] step 600 | zero indep-channels across batch: 34/42
[indep] step 700 | zero indep-channels across batch: 32/42
[indep] step 800 | zero indep-channels across batch: 32/42
[indep] step 900 | zero indep-channels across batch: 33/42
[indep] step 1000 | zero indep-channels across batch: 32/42
[indep] step 1100 | zero indep-channels across batch: 33/42
[indep] step 1200 | zero indep-channels across batch: 32/42
Epoch 01 | Train Acc 55.15% | Val Acc 71.56% | Part Presence F1 0.023
[indep] step 0 | zero indep-channels across batch: 33/42
[indep] step 100 | zero indep-channels across batch: 33/42
[indep] step 200 | zero indep-channels across 

KeyboardInterrupt: 

In [ ]:
num_labels = len(train_ds.supercats)
num_parts  = train_ds.num_part_channels

# Safely free any previous model
for k in list(globals().keys()):
    if k == 'model':
        del globals()[k]
torch.cuda.empty_cache()

model = SharedMTLResNet(num_labels=num_labels, num_parts=num_parts, use_sam=False).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
stopper = EarlyStopper(patience=3, min_delta=0.0)

for epoch in range(1, EPOCHS+1):
    tr_acc, tr_f1 = train_one_epoch(model, train_loader, opt)
    va_acc, va_f1 = evaluate(model, val_loader)

    print(f"Epoch {epoch:02d} | Train Acc {tr_acc*100:5.2f}% | Val Acc {va_acc*100:5.2f}% | Part Presence F1 {va_f1:.3f}")

    stopper.step(va_acc)
    if stopper.stop:
        print("Early stopped.")
        break

[dbg] step 0 | gt zero-channels (count over batch*channels): 8
[dbg] step 100 | gt zero-channels (count over batch*channels): 6
[dbg] step 200 | gt zero-channels (count over batch*channels): 4
[dbg] step 300 | gt zero-channels (count over batch*channels): 4
[dbg] step 400 | gt zero-channels (count over batch*channels): 6
[dbg] step 500 | gt zero-channels (count over batch*channels): 4
[dbg] step 600 | gt zero-channels (count over batch*channels): 4
[dbg] step 700 | gt zero-channels (count over batch*channels): 5
[dbg] step 800 | gt zero-channels (count over batch*channels): 6
[dbg] step 900 | gt zero-channels (count over batch*channels): 7
[dbg] step 1000 | gt zero-channels (count over batch*channels): 4
[dbg] step 1100 | gt zero-channels (count over batch*channels): 6
[dbg] step 1200 | gt zero-channels (count over batch*channels): 3
Epoch 01 | Train Acc 40.18% | Val Acc 11.86% | Part Presence F1 0.689
[dbg] step 0 | gt zero-channels (count over batch*channels): 4
[dbg] step 100 | gt z

## Integrating SAM

In [ ]:
from huggingface_hub import hf_hub_download

chkpt_path = hf_hub_download("ybelkada/segment-anything", "checkpoints/sam_vit_h_4b8939.pth")


In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_b"](checkpoint=chkpt_path)
sam.to(device)
sam_predictor = SamPredictor(sam)

# Wrap inference with SAM
class SAMWrapper:
    def __init__(self, predictor: SamPredictor, part_vocab: List[str]=PART_VOCAB):
        self.predictor = predictor
        self.part_vocab = part_vocab

    def predict_masks(self, image_pil: Image.Image, target_size=(224,224)) -> torch.Tensor:
        """
        Generate part-like masks from SAM. For now, SAM proposals are mapped heuristically.
        """
        image_np = np.array(image_pil)
        self.predictor.set_image(image_np)
        masks, scores, _ = self.predictor.predict(multimask_output=True)

        # Simple heuristic: distribute first N masks into part channels
        C = len(self.part_vocab)
        H, W = target_size
        mask_tensor = torch.zeros((C, H, W), dtype=torch.float32)

        for i, m in enumerate(masks):
            if i >= C:
                break
            m_resized = Image.fromarray(m.astype(np.uint8)*255).resize((W, H), resample=Image.NEAREST)
            mask_tensor[i] = torch.from_numpy(np.array(m_resized) > 127).float()

        return mask_tensor

# Usage in inference
sam_wrapper = SAMWrapper(sam_predictor)

model.eval()
with torch.no_grad():
    for i in range(5):
        img, label, _, presence, _ = val_ds[i]

        # Use SAM instead of GT masks
        sam_mask = sam_wrapper.predict_masks(T.ToPILImage()(img), target_size=IMG_SIZE).unsqueeze(0).to(device)
        img_t = img.unsqueeze(0).to(device)

        logits_y, logits_p = model(img_t, sam_mask)
        pred_label = logits_y.argmax(1).item()

        print(f"Image {i}: GT Label={label}, Pred Label={pred_label}, Parts Score={logits_p.sigmoid()[0,:5]}")

RuntimeError: Error(s) in loading state_dict for Sam:
	Unexpected key(s) in state_dict: "image_encoder.blocks.12.norm1.weight", "image_encoder.blocks.12.norm1.bias", "image_encoder.blocks.12.attn.rel_pos_h", "image_encoder.blocks.12.attn.rel_pos_w", "image_encoder.blocks.12.attn.qkv.weight", "image_encoder.blocks.12.attn.qkv.bias", "image_encoder.blocks.12.attn.proj.weight", "image_encoder.blocks.12.attn.proj.bias", "image_encoder.blocks.12.norm2.weight", "image_encoder.blocks.12.norm2.bias", "image_encoder.blocks.12.mlp.lin1.weight", "image_encoder.blocks.12.mlp.lin1.bias", "image_encoder.blocks.12.mlp.lin2.weight", "image_encoder.blocks.12.mlp.lin2.bias", "image_encoder.blocks.13.norm1.weight", "image_encoder.blocks.13.norm1.bias", "image_encoder.blocks.13.attn.rel_pos_h", "image_encoder.blocks.13.attn.rel_pos_w", "image_encoder.blocks.13.attn.qkv.weight", "image_encoder.blocks.13.attn.qkv.bias", "image_encoder.blocks.13.attn.proj.weight", "image_encoder.blocks.13.attn.proj.bias", "i